# AtlasVision tutorial: ask images questions in English, Igbo, Yoruba and Hausa

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/testatlassdk/blob/main/notebooks/atlasvision_tutorial.ipynb)

In this notebook you will:

1. install the `testatlasvision` package from PyPI,
2. load the model (about 10 minutes the first time),
3. ask questions about an image, in English and in Igbo, Yoruba or Hausa,
4. hold a conversation with follow-up questions,
5. learn the settings and limits worth knowing.

**What you need**
- A **GPU runtime**: *Runtime > Change runtime type > T4 GPU*.
- A **Hugging Face account with access to the gated model** [NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS). Request access on that page and wait for approval.
- A Hugging Face **access token** (a *Read* token is enough): <https://huggingface.co/settings/tokens>.
  Add it as a Colab Secret: click the key icon in the left sidebar, *Add new secret*, name it `HF_TOKEN`, paste the token and switch on *Notebook access*.
  (No secret? Step 1 will ask you to paste the token instead. Never share your token with anyone.)

Run the cells from top to bottom.

## 1. Set up

In [ ]:
# torchao conflicts with transformers/peft on Colab, so remove it first.
!pip uninstall -y -q torchao
!pip install -q "testatlasvision[4bit]"

In [ ]:
import torch
from testatlasvision.auth import resolve_token

assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

if resolve_token():
    print("Hugging Face token found.")
else:
    import os
    from getpass import getpass

    os.environ["HF_TOKEN"] = getpass("No HF_TOKEN secret found. Paste your Hugging Face token (hidden): ")

## 2. Load the model

This downloads about 16 GB the first time, so it takes several minutes. Colab forgets everything when the session ends,
so you repeat this each new session. Load the model **once** and reuse it for every cell below.

In [ ]:
from testatlasvision import (
    AtlasVisionError,
    GenerationConfig,
    TestAtlasVision,
)

model = TestAtlasVision()   # finds your token automatically
print(model.info())

## 3. Pick an image

Use the default image, paste any image address, or tick the box to upload a photo from your computer.

In [ ]:
#@title Choose an image
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"  #@param {type:"string"}
upload_my_own = False  #@param {type:"boolean"}

import testatlasvision

if upload_my_own:
    from google.colab import files

    uploaded = files.upload()
    image = testatlasvision.load_image(uploaded[next(iter(uploaded))])
else:
    image = testatlasvision.load_image(image_url)

display(image.resize((320, int(320 * image.height / image.width))))

## 4. Ask a question in English

`ask` answers one English question about the image. `describe` writes a caption.

In [ ]:
print("Q: What is in this image?")
print("A:", model.ask(image, "What is in this image?", config=GenerationConfig(max_new_tokens=80)))
print()
print("Caption:", model.describe(image, detailed=False))

## 5. Ask in Igbo, Yoruba or Hausa

Pick a language, type your question **in that language**, and run the cell. Change the values and run it again.

Behind the scenes `chat` translates your question to English, answers, and translates the answer back.
That is up to three generations, so it takes longer than English. The English version of the question and answer is
returned too, so you can check what the model understood.

Example questions: `Kedu ihe dị na foto a?` (Igbo), `Kí ni ó wà nínú àwòrán yìí?` (Yoruba), `Me ke cikin wannan hoton?` (Hausa).

In [ ]:
#@title Ask your own question
question = "Kedu ihe dị na foto a?"  #@param {type:"string"}
language = "Igbo"  #@param ["English", "Igbo", "Yoruba", "Hausa"]

result = model.chat(image, question, lang=language, config=GenerationConfig(max_new_tokens=150))

print(f"Question ({language}): {question}")
print(f"Answer   ({language}): {result.answer}")
print()
print("What the model saw in English:")
print("  question:", result.english_question)
print("  answer  :", result.english_answer)

### Describe the image in all four languages

In [ ]:
for language in ["English", "Igbo", "Yoruba", "Hausa"]:
    caption = model.describe(image, lang=language, detailed=False, config=GenerationConfig(max_new_tokens=100))
    print(f"[{language}] {caption}\n")

## 6. Have a conversation

A `Session` remembers earlier questions, so you can ask follow-ups such as "What colour is it?".
Choose a language, run the cell, type a question after `You:` and press Enter. Press Enter on an empty line to stop.

In [ ]:
#@title Conversation mode
conversation_language = "English"  #@param ["English", "Igbo", "Yoruba", "Hausa"]

session = model.new_session()
print("Ask about the image. Press Enter on an empty line to stop.\n")

while True:
    user_question = input("You: ").strip()
    if not user_question:
        break
    reply = model.chat(image, user_question, lang=conversation_language, session=session)
    print("AtlasVision:", reply.answer, "\n")

print(f"Conversation ended after {len(session.history)} turn(s).")

## 7. Text only and translation

N-ATLaS also works without an image: `text` answers any prompt and `translate` converts between the four languages.

In [ ]:
print(model.text("What is a positron? Answer in two sentences.", config=GenerationConfig(max_new_tokens=100)))
print()
print("Yoruba:", model.translate("Good morning, how are you?", "yo"))
print("Hausa :", model.translate("Thank you very much.", "ha"))

## 8. Settings you can change

`GenerationConfig` controls how the model writes:

| Setting | What it does |
|---|---|
| `max_new_tokens` | Maximum length of the answer. Raise it if answers stop mid-sentence. |
| `temperature` | `0` (default) gives the same answer every time. Higher values, such as `0.7`, give more varied answers. |
| `top_p` | Used only when `temperature` is above 0. |
| `repetition_penalty` | Discourages repeated phrases (default `1.1`). |

In [ ]:
prompt = "Describe this image."

steady = GenerationConfig(max_new_tokens=80)                       # same answer every run
varied = GenerationConfig(max_new_tokens=80, temperature=0.8)      # different answer each run

print("Steady:", model.ask(image, prompt, config=steady))
print()
print("Varied:", model.ask(image, prompt, config=varied))

## 9. When something goes wrong

Every error the SDK raises on purpose inherits from `AtlasVisionError`, with a message that says what to do.

In [ ]:
attempts = {
    "unknown language": lambda: model.chat(image, "hello", lang="French"),
    "image that does not exist": lambda: model.ask("/no/such/photo.jpg", "What is this?"),
    "empty question": lambda: model.ask(image, "   "),
}

for name, attempt in attempts.items():
    try:
        attempt()
    except (AtlasVisionError, ValueError) as error:
        print(f"{name}: {type(error).__name__}\n  {error}\n")

## 10. Good to know

- **Speed:** on a Colab T4, an English question takes a few seconds. A non-English `chat` takes roughly 15 to 20 seconds because of the translation steps.
- **Cut-off answers:** raise `max_new_tokens`. Translated answers need more room than English ones; `chat` already doubles your limit for the translate-back step.
- **Translation quality varies by language.** If a translated answer looks wrong, check `result.english_answer` to see whether the model understood the image.
- **Sessions are stored in English**, so a conversation can mix languages.
- **Colab resets** after inactivity. You re-install and re-load the model in each new session.
- **Keep your token private.** Do not paste it into shared notebooks or screenshots.

## Next steps

- Read the [README](https://github.com/testatlassdk#readme) for the full API, including notes on building a web service.
- Package on PyPI: <https://pypi.org/project/testatlasvision/>.
- Found a problem? [Open an issue](https://github.com/testatlassdk/issues).